# Assignment 4: Neural Network

What I want to do: create a regression neural network.\
I need to store the values of all gradients, loss, z, y, b and w for the first 10 steps.\
To do this will need some iteration going through storing each value for the variables.\
I then want to gather my loss equation and plot how it evolves.\
I need to do this twice, one for $\eta$=0.1 and one for $\eta$>0.1

In [46]:
# We need numpy to handle data and array
import numpy as np

# We need matplotlib.pyplot to make plots
import matplotlib.pyplot as plt\

from tabulate import tabulate

In [ ]:
# setting up the equations using the equations used in class
''' Activation Function '''
def sigmoid(x):
    '''
    sigmoid(x) = 1 / 1+exp(-x)
    '''
    return 1 / (1+np.exp(-x))

# loss function
def loss_func(y_predicted, y_results):
    ''' 
    Calculates the individual losses for each neuron
    ----------------
    Inputs
        - y_predicted: scalar
        - y_results: scalar
    '''
    return (y_predicted - y_results)**2


def weights_bias(w, eta, dL):
    ''' 
    same functional shape for both bias and weights
    w: weight or bias
    dL: either dL(w) or dL(b')
    -----------------
    Inputs
        - w: array
        - eta: scalar
        - dL: array
    '''
    # output should be an array (apart from bias_6 - has scalar input)
    return w - eta*dL


''' Defining the gradients'''
def forward(x1, x2, w11, w12, w1, b1, b6):
    ''' calculates the feed forward values, using a sigmoid activation function on the hidden layer and a reLU AF on the
    output layer. 
    Does this for all 5 neurons
    Returns what is essentially z_1, y_1, z_2 and y_2 from the added calculations in the document. 
    -----------------------
    Inputs:
        - x1, x2: scalar
        - w11, w12, b1: arrays of length 5, represent all input weights and biases
        - w1: array of 5, is output weights
        - b6: scalar, final bias of all neurons together
    '''
    # calculating the hidden layer:
    z_h = w11*x1 + w12*x2 + b1
    y_h = sigmoid(z_h)          # y = sigma(z) for all neurons

    # calculating the output:
    z_out = np.sum(w1*y_h) + b6 # will be scalar: is the sum of hidden layers for one step for each neuron
    y_out = z_out # the predicted y

    # outputs are: array, array, scalar, scalar
    return z_h, y_h, z_out, y_out

def gradients(y_hidden, y_predicted, y_result, x1, x2, w1):
    '''
    calculates the gradients of all the neurons for said step.
    -----------------------
    Inputs
        - y_hidden: array -> y_1
        - y_predicted: scalar inputs -> y_2
        - y_results: scalar input -> is what we are training after
        - x1, x2: scalar inputs
        - w1: array -> is output weight
    '''
    # gradients
    # output neuron - reLU
    dLdy2 = 2*(y_predicted - y_result)          
    # output layer neuron
    dLdb6 = dLdy2                               
    dLdw1 = dLdy2*y_hidden 
    
    # input layer neurons - sigmoid
    dLdb1 = dLdy2 * w1 * (1 - y_hidden) * y_hidden
    dLdw11 = dLdb1*x1
    dLdw12 = dLdb1*x2
    
    return {'dLdy2': dLdy2, 
            'dLdw1': dLdw1, # array
            'dLdb6': dLdb6, # scalar
            'dLdw11': dLdw11, # array
            'dLdw12': dLdw12, # array
            'dLdb1': dLdb1} # array


Initialising variables

In [64]:
steps = np.arange(0,10)
neurons = np.array([0,1,2,3,4]) #it is 1-5 however for iteration purposes this was the easiest way
total_loss = []

# initial values
x_1 = steps
x_1 = (x_1 - np.mean(x_1)) / np.std(x_1) # upon suggestion from AI as loss was not decreasing
x_2 = np.arange(1.5,11.5)
x_2 = (x_2 - np.mean(x_2)) / np.std(x_2)

# change value here for eta and run the subsequent cells
eta = 0.6

# initial function we aim for
y_results = 2*steps + 3

# as initial weights and biases per path is different
w_11 = np.array([0.5, 0.4, 0.3, 0.5, 0.7])
w_12 = np.array([0.4, 0.4, 0.8, 0.3, 0.5])
w_1 = np.array([0.8, 0.9, 0.6, 0.5, 0.8])
b_1 = np.array([0, 1, 2, 3, 4])
b_6 = 0

grads = {
    'dLy2': [],
    'dLw1': [],
    'dLb6': [],
    'dLw11': [],
    'dLw12': [],
    'dLb1': []
}

Starting the iteration through the required 10 steps

In [65]:
# as we are not doing over a large number of steps, a simple for loop should be sufficient
results = []
total_loss = []
for i in steps:

    #need to save the initial values present:
    w11_int = w_11.copy()
    w12_int = w_12.copy()
    w1_int = w_1.copy()
    b1_int = b_1.copy()
    b6_int = b_6

    #feed forward
    z_hid, y_hid, z_out, y_out = forward(x_1[i], x_2[i], w_11, w_12, w_1, b_1, b_6)

    # loss for this sample:
    losses = loss_func(y_out, y_results[i])
    total_loss.append(losses)

    # gradients
    gradient = gradients(y_hid, y_out, y_results[i], x_1[i], x_2[i], w_1)

    #storing it all before updating
    results.append({
        'Step': i+1,
        'w11': w11_int.copy(),
        'w12': w12_int.copy(),
        'b1': b1_int.copy(),
        'z_hidden': z_hid.copy(),
        'y_hidden': y_hid.copy(),
        'dLdw11': gradient['dLdw11'].copy(),
        'dLdw12': gradient['dLdw12'].copy(),
        'dLdb1': gradient['dLdb1'].copy(),
        'w1': w1_int.copy(),
        'b6': b6_int,
        'z_out': z_out,
        'y_predicted': y_out,
        'dLdw1': gradient['dLdw1'].copy(),
        'dLdb6': gradient['dLdb6'],
        'Loss': losses
    })

    # calculating the loss, weights and biases - updating the parameters
    w_11 = weights_bias(w_11, eta, gradient['dLdw11']) 
    w_12 = weights_bias(w_12, eta, gradient['dLdw12'])
    w_1 = weights_bias(w_1, eta, gradient['dLdw1'])
    b_1 = weights_bias(b_1, eta, gradient['dLdb1'])
    b_6 = weights_bias(b_6, eta, gradient['dLdb6'])

mean_loss = np.mean(total_loss)
print("mean loss: ", mean_loss)

mean loss:  290.03987373091456


In [ ]:
# this full block is entirely AI - it gave a set-up that allowed me to tabulate to see the final outputs of my network
table = []

for r in results:
    row = [
        r['Step'],
        r['y_predicted'],
        r['Loss'],
        r['dLdb6']
    ]

    # Append each hidden neuron's values
    row.extend(r['w11'])
    row.extend(r['w12'])
    row.extend(r['dLdw11'])
    row.extend(r['dLdw12'])
    row.extend(r['dLdb1'])
    row.extend(r['dLdw1'])

    table.append(row)

headers = (
    ['Step', 'Prediction', 'Loss', 'dLdb6']
    + [f'w{j+1}1' for j in neurons]
    + [f'w{j+1}2' for j in neurons]
    + [f'dLdw_{j+1}1' for j in neurons]
    + [f'dLdw_{j+1}2' for j in neurons]
    + [f'dLdb_{j+1}' for j in neurons]
    + [f'dLdw_{j+1}' for j in neurons]
)

print(tabulate(
    table,
    headers=headers,
    tablefmt='grid',
    floatfmt='.4f'
))

+---------+--------------+-----------+----------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+
|    Step |   Prediction |      Loss |    dLdb6 |     w11 |     w21 |     w31 |     w41 |     w51 |     w12 |     w22 |     w32 |     w42 |     w52 |   dLdw_11 |   dLdw_21 |   dLdw_31 |   dLdw_41 |   dLdw_51 |   dLdw_12 |   dLdw_22 |   dLdw_32 |   dLdw_42 |   dLdw_52 |   dLdb_1 |   dLdb_2 |   dLdb_3 |   dLdb_4 |   dLdb_5 |   dLdw_1 |   dLdw_2 |   dLdw_3 |   dLdw_4 |   dLdw_5 |
+=========+==============+===========+==========+=========+=========+=========+=========+=========+=========+=========+=========+=========+=========+===========+===========+===========+===========+===========+===========+===========+=======

This assignment has had the use of AI to help. If code has been explicitly copied it has been noted.\
It was utilised to help debug and fix misunderstanding issues as initially everything was written in a specific way for the way I was calculating the neurons upon which I found I had done wrong and needed help reformatting some elements.\
The code logic and formatting remains my own.